## 03 - Pitt dataset cleaning
In this notebook, we will try to detect overlaps between Pitt dataset and our train / test dataset

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

In [2]:
# load Pitt dataset

# src_fname = "../data/splits/train_50pc.csv"
src_fname = "../data/splits/train_pitt.csv"

src_df = pd.read_csv(src_fname)
src_df.head()

,file_id,text,target
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD
2,003-0,here's a cookie jar .\nand the lid is off the ...,AD
3,005-0,the boy is slipping off the stool .\nhe's tryi...,AD
4,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD


In [3]:
# load original test set
# tgt_fname = "../data/splits/train_pitt.csv"
tgt_fname = "../data/splits/test_text.csv"
# tgt_fname = "../data/splits/train_50pc.csv"

tgt_df = pd.read_csv(tgt_fname)
tgt_df.head()

,ID,Prediction,text
0,adrsdt15,NaN,"I see oh boy, almost falling on a stool trying..."
1,adrsdt40,NaN,Well the kids are trying to get the cookies an...
2,adrsdt26,NaN,"Okay, this is a family setting in a kitchen an..."
3,adrsdt67,NaN,"Oh, what do I see in this picture? This remind..."
4,adrsdt58,NaN,While the young boy is reaching for a cookie j...


In [16]:
# get a random item from src dataset
import random
# k = random.randint(0, len(src_df))
k = 184
print(k)
src_item = src_df.iloc[k]
src_item["text"]

184


"mhm .\noh .\nthere's a cookie jar .\nand the boy is toppling off (.) a stool .\nand the mother is spilling the water .\nand the\n(..) that's about it ."

In [17]:
# calculate embeddings
# model = SentenceTransformer("all-distilroberta-v1")
# model = SentenceTransformer("all-mpnet-base-v2")
# model = SentenceTransformer("sentence-T5-large")
model = SentenceTransformer("all-roberta-large-v1") # the same model as in hybrid approach
s = "Hello world"
v = model.encode(s)
print(v.shape)

(1024,)


In [18]:
# apply to src item
src_v = model.encode(src_item["text"])
print(src_v.shape)

(1024,)


In [19]:
# apply embeddings to target df
print("Calculating target embeddings...")
tgt_vecs = model.encode(tgt_df["text"])
print("Completed")
print(tgt_vecs.shape)

Calculating target embeddings...
Completed
(71, 1024)


In [21]:
def compare(src_v, tgt_vecs, threshold=0.85, quiet=True):
    """
        compare source vector (1 vector) with a list of target vectors
        return the index of most similar vector
    """
    # result = []
    result = None
    max_score = 0
    max_id = None
    
    for i, v in enumerate(tgt_vecs):
        score = util.cos_sim(v, src_v).item()
        if not quiet:
            print("Score =", score)
        if  score >= threshold and score > max_score:
            print("New high similarity detected. Index =", i)
            # result.append(i)
            max_score = score
            max_id = i
    
    
    # return result
    return max_id

result = compare(src_v, tgt_vecs, threshold=0.6, quiet=False)
# print("Counts =", len(result))
print(result)

Score = 0.6053565740585327
New high similarity detected. Index = 0
Score = 0.5803213715553284
Score = 0.460700660943985
Score = 0.32941585779190063
Score = 0.5687123537063599
Score = 0.3492952585220337
Score = 0.4630916714668274
Score = 0.4525010585784912
Score = 0.5858467221260071
Score = 0.43810001015663147
Score = 0.6144494414329529
New high similarity detected. Index = 10
Score = 0.6501197218894958
New high similarity detected. Index = 11
Score = 0.6110761165618896
Score = 0.38539186120033264
Score = 0.37243109941482544
Score = 0.4850243926048279
Score = 0.5340250134468079
Score = 0.4943351447582245
Score = 0.5243884921073914
Score = 0.5144447684288025
Score = 0.5652144551277161
Score = 0.6651232242584229
New high similarity detected. Index = 21
Score = 0.49976399540901184
Score = 0.598824143409729
Score = 0.5675228238105774
Score = 0.5264087915420532
Score = 0.5760680437088013
Score = 0.5629412531852722
Score = 0.5771655440330505
Score = 0.5891528725624084
Score = 0.47759786248207

In [9]:
# ### Now we can do pair-wise checking
# ID_field = "file_id" # set to ID for original test set

# result = []
# for i in range(len(src_df)):
#     print("Checking item =", i)
#     item = src_df.iloc[i]
#     src_text = item["text"]
#     src_v = model.encode(src_text)
#     sim_id = compare(src_v, tgt_vecs, threshold=0.95)
#     # if len(sim_id) > 0:
#     if sim_id is not None:
#         print("High similarity detected")
#         src_file_id = item[ID_field]
#         print(f"Src id = {i}, file_id = [{src_file_id}], \ntext = [{src_text}]")
#         print(f"Similar ids = {sim_id}")
#         for j in sim_id:
#             tgt_text = tgt_df.iloc[j]["text"]
#             tgt_file_id = tgt_df.iloc[j]["file_id"]
#             print("Target text =", tgt_text)
#             print("-------")
#             result.append((src_text, tgt_text, src_file_id, tgt_file_id))
                  
    

In [35]:
### Now we can do pair-wise checking
# ID_field = "file_id" # set to ID for original test set
src_ID_field = "file_id" # set to ID for original test set
tgt_TD_field = "ID"
threshold = 0.75

result = []
for i in range(len(src_df)):
    print("Checking item =", i)
    item = src_df.iloc[i]
    src_text = item["text"]
    src_v = model.encode(src_text)
    sim_id = compare(src_v, tgt_vecs, threshold=threshold)
    # if len(sim_id) > 0:
    if sim_id is not None:
        print("High similarity detected")
        src_file_id = item[src_ID_field]
        print(f"Src id = {i}, file_id = [{src_file_id}], \ntext = [{src_text}]")
        print(f"Similar ids = {sim_id}")
        # for j in sim_id:
        print("--------")
        tgt_text = tgt_df.iloc[sim_id]["text"]
        tgt_file_id = tgt_df.iloc[sim_id][tgt_TD_field]
        print("Target text =", tgt_text)
        print("-------")
        result.append((src_text, tgt_text, src_file_id, tgt_file_id))
                  
    

Checking item = 0
Checking item = 1
Checking item = 2
Checking item = 3
Checking item = 4
Checking item = 5
Checking item = 6
Checking item = 7
Checking item = 8
Checking item = 9
Checking item = 10
Checking item = 11
Checking item = 12
Checking item = 13
Checking item = 14
Checking item = 15
Checking item = 16
Checking item = 17
Checking item = 18
Checking item = 19
Checking item = 20
Checking item = 21
Checking item = 22
Checking item = 23
New high similarity detected. Index = 38
High similarity detected
Src id = 23, file_id = [029-1], 
text = [alright .
I see the little boy stealing cookies from the cookie jar .
and <the little girl's>  &-uh he gave some to the little girl .
and she's eatin(g) some of the cookies .
and I guess this is mama .
and she's washing the dishes .
and she dropped a dish .
no  no ‡ she didn't drop a dish .
uh the  the water that she's washing the dishes with she let
and it's overflown  &=laughs &+o &+o overflown .
that doesn't>
did &*INV:no it &-uh +/?
we for

In [36]:
print("Count =", len(result))
print("All similar text pairs:")
for (text1, text2, src_file_id, tgt_file_id) in result:
    print(f"ID={src_file_id}, Source text =", text1[:300])
    print("--------")
    print(f"ID={tgt_file_id}, Target text =", text2[:300])
    print("================================")

Count = 68
All similar text pairs:
ID=029-1, Source text = alright .
I see the little boy stealing cookies from the cookie jar .
and <the little girl's>  &-uh he gave some to the little girl .
and she's eatin(g) some of the cookies .
and I guess this is mama .
and she's washing the dishes .
and she dropped a dish .
no  no ‡ she didn't drop a dish .
uh the  
--------
ID=adrsdt20, Target text = All right, tell me what you see going on in that picture. I see the little boy stealing cookies from the cookie jar. And the little girls and he gave some to the little girl and she's eat some of the cookies. And I guess this is mama and she's washing the dishes. And she dropped a dish. No, no, she 
ID=030-0, Source text = it's a picture of a kitchen .
there's a child reaching on  a cookie jar .
um <a male>
he's falling .
the stool is falling .
uh his  a female youngster maybe sister is
and she's got a finger to her lip .
the &-uh mother is at the sink washing a dish .
the water is spilling over f

In [37]:
# mark on the source (Pitt dataset) which rows is likely to included in the target set (orginal test set)
src_df["possible_overlap"] = src_df["text"].apply(lambda text: 
                                                  1 if (compare(model.encode(text), 
                                                          tgt_vecs, 
                                                          threshold=threshold
                                                        )) 
                                                  else 0
                                                 )
src_df.head()

New high similarity detected. Index = 38
New high similarity detected. Index = 61
New high similarity detected. Index = 66
New high similarity detected. Index = 39
New high similarity detected. Index = 55
New high similarity detected. Index = 43
New high similarity detected. Index = 11
New high similarity detected. Index = 55
New high similarity detected. Index = 1
New high similarity detected. Index = 46
New high similarity detected. Index = 68
New high similarity detected. Index = 44
New high similarity detected. Index = 60
New high similarity detected. Index = 67
New high similarity detected. Index = 38
New high similarity detected. Index = 38
New high similarity detected. Index = 60
New high similarity detected. Index = 46
New high similarity detected. Index = 0
New high similarity detected. Index = 38
New high similarity detected. Index = 58
New high similarity detected. Index = 48
New high similarity detected. Index = 55
New high similarity detected. Index = 21
New high similarit

,file_id,text,target,possible_overlap
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD,0
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD,0
2,003-0,here's a cookie jar .\nand the lid is off the ...,AD,0
3,005-0,the boy is slipping off the stool .\nhe's tryi...,AD,0
4,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD,0


In [38]:
src_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549 entries, 0 to 548
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   file_id           549 non-null    object
 1   text              549 non-null    object
 2   target            549 non-null    object
 3   possible_overlap  549 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 17.3+ KB


In [39]:
# count how many possible items in Pitt dataset is contained in the test set
count = len(src_df.query("possible_overlap == 1"))
print("Count of possible overlaps in test set =", count)

Count of possible overlaps in test set = 68


In [40]:
# export back to the source (train_pitt.csv)
output_write = True

if output_write:
    fname_out = "../data/splits/train_pitt_markedB.csv"
    print("Output to ", fname_out)
    src_df.to_csv(fname_out, index=False)
    print("Completed")

Output to  ../data/splits/train_pitt_markedB.csv
Completed
